## Token Intervention with Fixed Reasoning

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

### Set-up

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _util

In [3]:
model_type = "GPT-OSS" # GPT-OSS or R1

if model_type == "GPT-OSS":
    model, tokenizer = _util.load_OSS()
elif model_type == "R1":
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
prompt_type = "h_pre_result_2" # empty or pre_result or pre_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    divided_prompts = pd.read_csv(f"data/{model_type}/h_divided_prompts{prompt_type[2:]}.csv")
else:
    divided_prompts = pd.read_csv(f"data/{model_type}/divided_prompts{prompt_type}.csv")
divided_prompts["base_number"] = divided_prompts["base_number"].astype('Int64')
divided_prompts["source_number"] = divided_prompts["source_number"].astype('Int64')
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 2816 divided prompts


In [8]:
def get_fixed_reasoning_intervention_prompt(row):
    return row['base_before'] + str(row['source_number']) + row['base_after']

In [11]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['intervention_prompt', 'generated_text']

filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}", f"fixed_reasoning{prompt_type}.csv", header, overwrite=True)

batch_size = 22

for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    intervention_prompts = [get_fixed_reasoning_intervention_prompt(row) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokenized_inputs = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    # Generate for the entire batch
    generations = model.generate(
        tokenized_inputs.input_ids,
        attention_mask=tokenized_inputs.attention_mask,
        max_new_tokens=3,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(generations[j]).replace(intervention_prompt, "").replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_prompt, generated_text])


  0%|                                                                                         | 0/128 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████| 128/128 [08:43<00:00,  4.09s/it]
